# Moral Hazard Workshop: Estimating a Structural Principal-Agent Model
### Student Version

Estimate the static principal-agent moral hazard model from Margiotta & Miller (2000) on a simulated dataset whose true parameters are withheld from you, then
submit `submission.csv` for scoring.

Methods marked `# TODO` below are the pieces you fill in yourself:

1. $\hat\psi$ (Step 1)
2. the truncated-normal MLE for $(\hat\mu_w,\hat\sigma)$ (Step 2)
3. `compute_g`
4. `optimal_wage`
5. the NLS fitting call in Step 3
6. part of `compute_cost_decomposition` (Delta1 and Delta3 -- Delta2 is given
   as a worked example)
7. `run_counterfactual`'s cost recomputation -- the same formulas from (6),
   just reapplied to rescaled parameters

Everything else -- including the bootstrap standard errors and
`make_submission`/`validate_submission` -- is provided as working reference
code. You're welcome to read through it, but you don't need to write or
debug any of it.

In [1]:
# ============================================================================
# IMPORTS
# ============================================================================
import numpy as np
import pandas as pd
from scipy import stats, optimize
import time
import os

pd.set_option('display.float_format', lambda x: f"{x:,.6f}")

In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_PATH = '/kaggle/input/competitions/dse2026-practicum-2/simulated_moral_hazard_data.dta'                  

# Input data files are available in the read-only "../input/" directory.
# This lists every file the competition attached to your session.
for dirname, _, filenames in os.walk(DATA_PATH):
    for filename in filenames:
        print(os.path.join(dirname, filename))

X_VAR = 'net_excess_ret'   # performance measure (net excess return)
W_VAR = 'totalComp'        # compensation

## The Model

This is the static principal-agent model with moral hazard from Margiotta &
Miller (2000) and the accompanying lecture slides. You are not deriving any
of this -- it's given in the Kaggle Competition overview, so you know exactly what you're estimating.

**Estimation, in three steps, plus standard errors:**
1. $\hat\psi = \min_i x_i$ (truncation point)
2. $(\hat\mu_w,\hat\sigma)$: maximum likelihood on the truncated normal, using only $\{x_i\}$
3. $(\hat\gamma,\hat\alpha,\hat\beta,\hat\mu_s)$: nonlinear least squares, minimizing $\sum_i (w_i - w^*(x_i))^2$, with $\eta$ solved as an inner step at every trial parameter vector.
   Note: the optimizer's free parameters are $(\gamma,\alpha,r,\mu_s)$ with $r=\alpha/\beta$, not $(\gamma,\alpha,\beta,\mu_s)$ -- this is a numerical-conditioning choice (see the Step 3 docstring), not a modeling one. $\beta=\alpha/r$ is recovered afterward and is the quantity reported and submitted; $r$ itself is never a quantity of interest.
8. Standard errors on all of the above, via bootstrap (see `bootstrap_se`).


## The `MoralHazardModel` class

Steps 1-2, `compute_g`, `optimal_wage`, the NLS call in Step 3, part of
`compute_cost_decomposition` (Delta1 and Delta3 -- Delta2 is given as a
worked example), and `run_counterfactual`'s cost recomputation (the same
formulas, just reapplied to rescaled parameters) are the pieces you
implement yourself (marked `# TODO` below). Everything else -- the
multi-start search machinery around the Step 3 NLS call, `solve_eta`, the
bootstrap standard errors, and `make_submission`/`validate_submission` -- is
provided as working reference code. Robust optimization, a subtle piece of
bootstrap theory, and correctly-formatted output are numerical/statistical
plumbing, not the economics being tested here -- you're welcome to read
through them, but you don't need to write or debug any of it.

In [3]:
# ============================================================================
# MoralHazardModel -- student version. Methods marked TODO are yours to
# fill in; everything else is provided, working reference code.
# ============================================================================

from typing import Dict, Tuple, Optional


def _tn_mean(mu, sigma, psi):
    """Mean of a normal(mu, sigma^2) truncated below at psi."""
    a = (psi - mu) / sigma
    return mu + sigma * np.exp(stats.norm.logpdf(a) - stats.norm.logsf(a))


class MoralHazardModel:
    """Sequential estimator for the static moral-hazard model."""

    def __init__(self):
        self.x = None
        self.w = None
        self.n = None
        self.psi = None
        self.mu_w_hat = None
        self.sigma_hat = None
        self.gamma_hat = None
        self.alpha_hat = None
        self.beta_hat = None
        self.mu_s_hat = None
        self.eta_hat = None
        self.ssr_ = None

    def set_data(self, x, w):
        self.x = np.asarray(x, dtype=float).flatten()
        self.w = np.asarray(w, dtype=float).flatten()
        self.n = len(self.x)
        print(f"Data loaded: n = {self.n}")
        print(f"  x: mean={self.x.mean():.4f}, std={self.x.std():.4f}, min={self.x.min():.4f}")
        print(f"  w: mean={self.w.mean():,.2f}, std={self.w.std():,.2f}")

    # ------------------------------------------------------------------
    # STEP 1: truncation threshold  <-- TODO
    # ------------------------------------------------------------------
    def step1_estimate_threshold(self) -> float:
        """
        ******************** TODO *****************************
        Step 1: Estimate truncation threshold psi.

        The truncation point is the minimum observed output.

        Returns:
        --------
        psi : float
        Estimated truncation threshold
        """
        # ==================== YOUR CODE HERE ====================
        raise NotImplementedError("TODO: implement Step 1 (see docstring above)")
        # ==========================================================
        return self.psi

    # ------------------------------------------------------------------
    # STEP 2: working-distribution MLE, truncated normal  <-- TODO
    # ------------------------------------------------------------------
    def step2_estimate_f(self) -> Tuple[float, float]:
        """
        ******************** TODO *****************************
        Step 2: Estimate parameters of f(x).

        Estimate (mu, sigma) via Limited Information Maximum Likelihood
        (LIML) on the truncated normal distribution.

        The log-likelihood for truncated normal TN(μ, σ²; ψ) is:
        ℓ(μ,σ) = Σ log φ((x_i - μ)/σ) - n log σ - n log(1 - Φ((ψ-μ)/σ))

        where φ is standard normal PDF and Φ is standard normal CDF.
        
        Hint: write the negative log-likelihood above as a function of
        (mu, log_sigma) -- optimizing over log(sigma) instead of sigma keeps
        sigma > 0 automatically. The optimizer call below is already written
        for you -- you only need to fill in `negloglik`.

        Returns:
        --------
        mu_hat : float
            Estimated mean
        sigma_hat : float
            Estimated std
        """
        def negloglik(params):
            mu, log_sigma = params
            # ==================== YOUR CODE HERE ====================
            raise NotImplementedError("TODO: implement negloglik (see docstring above)")
            # ==========================================================

        x0 = [self.x.mean(), np.log(self.x.std())]
        res = optimize.minimize(negloglik, x0, method='Nelder-Mead',
                                 options={'xatol': 1e-10, 'fatol': 1e-10, 'maxiter': 5000})
        self.mu_w_hat = float(res.x[0])
        self.sigma_hat = float(np.exp(res.x[1]))
        return self.mu_w_hat, self.sigma_hat

    # ------------------------------------------------------------------
    # g(x): the likelihood ratio  <-- TODO
    # ------------------------------------------------------------------
    def compute_g(self, x, mu_s, mu_w=None, sigma=None, psi=None):
        """
        TODO: implement g(x) using the formula rewritten below:

            ln g(x) = ln Φ[(μ_w-ψ)/σ] - ln Φ[(μ_s-ψ)/σ]
                        + (μ_w**2 - μ_s**2)/(2*σ**2)
                        + (μ_s - μ_w)/σ**2 * x

          (μ_w, σ, and ψ defaulting to the fitted Step 1-2 values, but can be
        overridden)
        """
        mu_w = self.mu_w_hat if mu_w is None else mu_w
        sigma = self.sigma_hat if sigma is None else sigma
        psi = self.psi if psi is None else psi
        # ==================== YOUR CODE HERE ====================
        # Build log_g
        raise NotImplementedError("TODO: implement compute_g (see docstring above)")
        # ==========================================================
        return np.exp(np.clip(log_g, -50, 50))

    # ------------------------------------------------------------------
    # eta: unique positive root of the IC-binding condition (provided)
    # ------------------------------------------------------------------
    def solve_eta(self, r, g_vals) -> Tuple[float, bool]:
        """Solve E[h/(1+eta*h)]=0, h(x)=r-g(x). Returns (eta, binding);
        binding=False means IC cannot bind at any finite eta (h(x)>=0
        everywhere) -- the optimal contract is then the flat wage, eta=0."""
        h = r - g_vals
        hmin = h.min()
        if hmin >= 0:
            return 0.0, False
        upper = -1.0 / hmin
        lo, hi = 1e-14, upper * (1 - 1e-9)

        def eq(eta):
            return np.mean(h / (1 + eta * h))

        if eq(lo) * eq(hi) > 0:
            return 0.0, False
        eta = optimize.brentq(eq, lo, hi, xtol=1e-12, rtol=1e-12, maxiter=300)
        return float(eta), True

    # ------------------------------------------------------------------
    # w*(x): optimal wage schedule  <-- TODO
    # ------------------------------------------------------------------
    def optimal_wage(self, x, gamma, alpha, beta, mu_s, mu_w=None, sigma=None, psi=None):
        """
        TODO: implement w*(x) using the formula given above:

            w*(x) = (1/γ) * ln[ α * (1 + η*(α/β - g(x))) ]

        Hint: Use the previous two functions when coding this.
        """
        # ==================== YOUR CODE HERE ====================
        raise NotImplementedError("TODO: implement optimal_wage (see docstring above)")
        # ==========================================================
        return wstar, eta

    # ------------------------------------------------------------------
    # STEP 3: NLS estimation, TRF + multi-start (provided)
    # ------------------------------------------------------------------
    def _residuals(self, theta):
        gamma, alpha, r, mu_s = theta
        beta = alpha / r
        wstar, eta = self.optimal_wage(self.x, gamma, alpha, beta, mu_s)
        return self.w - wstar

    def _step3_search_space(self, n_starts: int = 5):
        """Provided plumbing, not something you need to tune: bounds and
        starting points for the Step 3 search.

        bounds = (lo, hi) for (gamma, alpha, r, mu_s). The bounds on r and
        mu_s aren't arbitrary -- r > 1 is exactly alpha > beta, and
        mu_s < mu_w_hat is exactly "shirking looks worse than working".

        starts: several deliberately spread-out (gamma, alpha, r, mu_s)
        starting guesses for the search in step3_estimate_contract. Purely a
        numerical-robustness measure (protects against the optimizer landing
        in a bad local optimum, or against one unlucky starting point).
        """
        lo = [1e-9, 1e-3, 1.0001, self.psi + 1e-3]
        hi = [1e-2, 1000.0, 100.0, self.mu_w_hat - 1e-3]
        starts = [
            [1e-4, 2.0, 1.5, self.mu_w_hat - 0.2],
            [1e-4, 5.0, 1.1, self.mu_w_hat - 0.4],
            [1e-5, 1.5, 2.0, self.mu_w_hat - 0.6],
            [1e-3, 10.0, 1.3, self.mu_w_hat - 0.1],
            [1e-4, 3.0, 1.8, self.mu_w_hat - 0.3],
        ][:n_starts]
        return (lo, hi), starts

    def step3_estimate_contract(self, n_starts: int = 5) -> Dict:
        """Nonlinear least squares for (γ, α, r=α/β, μ_s).

        Note: the optimizer estimates r=α/β directly, not β
        itself -- purely a numerical-conditioning choice."""
        bounds, starts = self._step3_search_space(n_starts)

        best = None
        for theta0 in starts:
            # ==================== YOUR CODE HERE ====================
            # TODO: this one optimizer call is the actual NLS step.
            # Call scipy.optimize.least_squares, starting
            # from theta0, with method='trf' and bounds=bounds. Assign the
            # result to `res` -- the rest of the loop below uses it. Feel
            # free to add extra arguments if you want to tighten convergence.
            raise NotImplementedError("TODO: call optimize.least_squares (see comment above)")
            # ==========================================================
            
            ssr = float(np.sum(res.fun ** 2))
            if best is None or ssr < best['ssr']:
                gamma, alpha, r, mu_s = res.x
                beta = alpha / r
                _, eta = self.optimal_wage(self.x, gamma, alpha, beta, mu_s)
                best = dict(gamma=gamma, alpha=alpha, beta=beta, r=r, mu_s=mu_s, eta=eta, ssr=ssr)

        self.gamma_hat, self.alpha_hat, self.beta_hat = best['gamma'], best['alpha'], best['beta']
        self.mu_s_hat, self.eta_hat, self.ssr_ = best['mu_s'], best['eta'], best['ssr']
        return best

    # ------------------------------------------------------------------
    # Cost of moral hazard: Delta1, Delta2, Delta3  <-- TODO
    # ------------------------------------------------------------------
    def compute_cost_decomposition(self) -> Dict:
        """
        TODO: implement Delta1, Delta2, and Delta3 using the formulas given
        above:

            Delta1 = E[ln(1 + eta*(alpha/beta - g(x)))] / gamma   (sample mean over observed x)
            Delta2 = ln(alpha/beta) / gamma  
            Delta3 = E[x|work] - E[x|shirk]   (use the provided _tn_mean helper)
        """
        gamma, alpha, beta, eta = self.gamma_hat, self.alpha_hat, self.beta_hat, self.eta_hat
        mu_w, mu_s, sigma, psi = self.mu_w_hat, self.mu_s_hat, self.sigma_hat, self.psi
        r = alpha / beta

        # ==================== YOUR CODE HERE ====================
        # Fill in g_vals (via self.compute_g), delta1, delta2, and delta3. Keep the
        # `return` line below.
        raise NotImplementedError("TODO: implement Delta1, Delta2, and Delta3 (see docstring above)")
        # ==========================================================

        return {'Delta1': delta1, 'Delta2': delta2, 'Delta3': delta3}

    # ------------------------------------------------------------------
    # Bootstrap standard errors (Step 8) -- fully provided reference code.
    # You don't need to implement or debug anything here.
    # ------------------------------------------------------------------
    def _bootstrap_one_replicate(self, x_b, w_b, psi_fixed, n_starts: int = 2) -> Optional[Dict]:
        """Run ONE bootstrap replicate.

        Given a resampled (x_b, w_b) pair (drawn with replacement from the
        original sample) and the full-sample psi_hat as `psi_fixed`, this
        re-estimates Steps 2-3 on the resampled data but holds psi fixed at
        the full-sample value in every replicate, rather than re-estimating
        it from the resample.

        Why: psi_hat = min(x) is an extremum estimator with a much faster
        convergence rate, and a non-normal limiting distribution, than
        mu_w_hat, sigma_hat, gamma_hat, alpha_hat, r_hat, mu_s_hat. The
        ordinary nonparametric bootstrap (resample-and-recompute) does not
        give a valid standard error for an estimator of that kind -- so it's
        held fixed instead, which is valid precisely because it converges so
        much faster than the other parameters that treating it as known
        doesn't bias their asymptotic standard errors. You don't need to
        reproduce this reasoning yourself -- it's here for context.

        Returns a dict with mu_w, sigma, mu_s, gamma, alpha, beta, r, eta,
        delta_1, delta_2, delta_3, r_at_lower_bound -- or None if this
        replicate doesn't produce a usable (finite) fit.
        """
        m = MoralHazardModel()
        m.x, m.w, m.n = x_b, w_b, len(x_b)
        m.psi = psi_fixed
        try:
            m.step2_estimate_f()
            fit = m.step3_estimate_contract(n_starts=n_starts)
            costs = m.compute_cost_decomposition()
        except Exception:
            return None

        out = dict(mu_w=m.mu_w_hat, sigma=m.sigma_hat, mu_s=m.mu_s_hat,
                   gamma=m.gamma_hat, alpha=m.alpha_hat, beta=m.beta_hat,
                   r=fit['r'], eta=m.eta_hat,
                   delta_1=costs['Delta1'], delta_2=costs['Delta2'], delta_3=costs['Delta3'],
                   r_at_lower_bound=bool(fit['r'] <= 1.0001 + 1e-6))
        check_vals = [out[k] for k in
                      ('mu_w', 'sigma', 'mu_s', 'gamma', 'alpha', 'beta', 'r', 'eta',
                       'delta_1', 'delta_2', 'delta_3')]
        if not np.all(np.isfinite(check_vals)):
            return None
        return out

    def bootstrap_se(self, B: int = 500, seed: int = 20260726, n_starts: int = 2) -> Dict:
        """Nonparametric bootstrap standard errors and 95% percentile CIs.

        Resamples (x_i, w_i) pairs with replacement and re-runs
        _bootstrap_one_replicate B times; that method decides which steps
        get re-estimated per replicate (see its docstring).

        B=500 implies a Monte Carlo floor on these SEs themselves of
        roughly SE_of_SE / SE ~ 1/sqrt(2B), about 3% here -- raise B if you
        need tighter SEs than that.

        Returns se_<name>/ci_lo_<name>/ci_hi_<name> for each of
        (mu_w, sigma, mu_s, gamma, alpha, beta, delta_1, delta_2, delta_3),
        plus se_eta/ci_*_eta and se_r/ci_*_r as diagnostics (not scored),
        n_used, n_failed, and r_at_lower_bound_share.

        There is no se_psi: psi is held fixed across replicates by design
        (see _bootstrap_one_replicate), and its own sampling distribution
        is not asymptotically normal, so a standard error would not be the
        right summary for it even if it were resampled.

        If total runtime is not tolerable, reduce n_starts here (not the
        n_starts used for the main Step 3 fit above) -- fewer starts per
        bootstrap replicate is a reasonable speed/robustness trade in a
        context where any one replicate's exact optimum matters much less
        than the main fit's does.
        """
        rng = np.random.default_rng(seed)
        records = []
        n_failed = 0
        for _ in range(B):
            idx = rng.integers(0, self.n, size=self.n)
            rep = self._bootstrap_one_replicate(self.x[idx], self.w[idx], self.psi, n_starts=n_starts)
            if rep is None:
                n_failed += 1
            else:
                records.append(rep)

        if not records:
            raise RuntimeError("bootstrap_se: every replicate failed -- check step3 convergence")

        out = {}
        for name in ['mu_w', 'sigma', 'mu_s', 'gamma', 'alpha', 'beta',
                     'delta_1', 'delta_2', 'delta_3', 'eta', 'r']:
            vals = np.array([rec[name] for rec in records], dtype=float)
            out[f'se_{name}'] = float(vals.std(ddof=1))
            out[f'ci_lo_{name}'] = float(np.percentile(vals, 2.5))
            out[f'ci_hi_{name}'] = float(np.percentile(vals, 97.5))

        out['n_used'] = len(records)
        out['n_failed'] = n_failed
        out['r_at_lower_bound_share'] = float(np.mean([rec['r_at_lower_bound'] for rec in records]))

        print(f"Bootstrap: {len(records)}/{B} replicates used ({n_failed} discarded for non-convergence).")
        print(f"Monte Carlo floor on these SEs at B={B}: ~{100/np.sqrt(2*B):.1f}% "
              f"(SE_of_SE/SE ~ 1/sqrt(2B)) -- raise B for tighter SEs.")
        print(f"Share of replicates with r at its lower bound (1.0001): "
              f"{out['r_at_lower_bound_share']:.1%}")
        if out['r_at_lower_bound_share'] > 0.05:
            print("WARNING: r hits its box constraint in a non-trivial share of replicates -- "
                  "percentile CIs for r/beta are not reliable there.")
        return out

    # ------------------------------------------------------------------
    # Counterfactual  <-- TODO
    # ------------------------------------------------------------------
    def run_counterfactual(self, gamma_mult=1.0) -> Dict:
        """TODO: recompute (Delta1,Delta2,Delta3) holding all else fixed at
        the fitted estimates, rescaling gamma. Resolve 
        components of model that are affected by gamma to determine how the costs of moral hazard change under increased risk adversion.
         """
        gamma = self.gamma_hat * gamma_mult
        psi = self.psi
        sigma = self.sigma_hat
        alpha, beta = self.alpha_hat, self.beta_hat
        mu_w, mu_s = self.mu_w_hat, self.mu_s_hat
        r = alpha / beta

        # ==================== YOUR CODE HERE ====================
        raise NotImplementedError("TODO: implement run_counterfactual's cost recomputation (see docstring above)")
        # ==========================================================
        return {'Delta1': delta1, 'Delta2': delta2, 'Delta3': delta3,
                'eta': eta, 'eta_binding': binding,
                'gamma': gamma}

    def summary(self):
        print("Step 1: psi     =", self.psi)
        print("Step 2: mu_w    =", self.mu_w_hat, " sigma =", self.sigma_hat)
        print("Step 3: gamma   =", self.gamma_hat)
        print("        alpha   =", self.alpha_hat)
        print("        beta    =", self.beta_hat)
        print("        mu_s    =", self.mu_s_hat)
        print("        eta     =", self.eta_hat)
        print("        (r = alpha/beta =", self.alpha_hat / self.beta_hat,
              " -- internal reparameterization, diagnostic only)")

## Estimation


In [4]:
# Load the data
df = pd.read_stata(DATA_PATH)
x = df[X_VAR].to_numpy(float)
w = df[W_VAR].to_numpy(float)

In [5]:
model = MoralHazardModel()
model.set_data(x, w)
model.step1_estimate_threshold()
model.step2_estimate_f()

t0 = time.time()
model.step3_estimate_contract()
print(f"\nFit time: {time.time()-t0:.2f}s\n")

model.summary()

Data loaded: n = 8000
  x: mean=-0.0077, std=0.6680, min=-1.4419
  w: mean=13,155.95, std=4,689.65


NotImplementedError: TODO: implement Step 1 (see docstring above)

### Cost of moral hazard

In [ ]:
costs = model.compute_cost_decomposition()
print(f"Delta1 (risk premium)      = {costs['Delta1']:,.2f}")
print(f"Delta2 (effort disutility) = {costs['Delta2']:,.2f}")
print(f"Delta3 (output loss, return units) = {costs['Delta3']:.6f}")

### Standard errors (Step 8)

Report the recovered structural parameters *and* their standard errors.
These come from a nonparametric bootstrap over Steps 2-3 (`bootstrap_se`,
fully provided -- you don't need to implement or debug this). Briefly: it
resamples the data with replacement, re-runs Steps 2-3 many times, and looks
at how much the estimates bounce around. One subtlety worth knowing about
even though it's not something you build: $\hat\psi$ is held fixed at its
full-sample value in every replicate rather than re-estimated, because it
converges to the truth much faster than the other parameters do -- see the
method's docstring if you're curious why.

In [ ]:
t0 = time.time()
boot = model.bootstrap_se(B=500)
print(f"\nBootstrap time: {time.time()-t0:.1f}s\n")
for name in ['mu_w', 'sigma', 'mu_s', 'gamma', 'alpha', 'beta', 'delta_1', 'delta_2', 'delta_3']:
    print(f"{name:>10}: SE={boot[f'se_{name}']:.6g}  "
          f"[{boot[f'ci_lo_{name}']:.6g}, {boot[f'ci_hi_{name}']:.6g}]")

## Counterfactual

This is where your `run_counterfactual` TODO gets exercised -- the same
Delta1/Delta2/Delta3 formulas from `compute_cost_decomposition`, just
reapplied at a rescaled gamma instead of the fitted one.

Holding everything else fixed at the fitted estimates, recompute the
optimal contract and the three cost measures under higher risk aversion:
$\gamma \to 2\gamma$.

In [ ]:
scenarios = {
    'gamma': {'gamma_mult': 2.0},
}

baseline = model.run_counterfactual()
print(f"{'Baseline':20s}  eta={baseline['eta']:.4f} (binding={baseline['eta_binding']})  "
      f"Delta1={baseline['Delta1']:,.2f}  Delta2={baseline['Delta2']:,.2f}  Delta3={baseline['Delta3']:.6f}")

grid = {}
for name, kwargs in scenarios.items():
    r = model.run_counterfactual(**kwargs)
    grid[name] = {'delta_1': r['Delta1'], 'delta_2': r['Delta2'], 'delta_3': r['Delta3']}
    print(f"{name:20s}  eta={r['eta']:.4f} (binding={r['eta_binding']})  "
          f"Delta1={r['Delta1']:,.2f}  Delta2={r['Delta2']:,.2f}  Delta3={r['Delta3']:.6f}")

## Submission

Write `submission.csv` for Kaggle scoring: one row per required `ID`, a
single `ESTIMATE` column, in the fixed order given by `SUBMISSION_IDS`
below. No `r` row (it's an internal reparameterization -- `beta` is the
submitted quantity, see the Step 3 note above) and no `se_psi` row
($\hat\psi$ is held fixed in the bootstrap by design, and its sampling
distribution isn't normal, so a standard error isn't the right summary for
it anyway).


In [ ]:
# ============================================================================
# Submission plumbing -- provided, no TODO
# ============================================================================

SUBMISSION_IDS = [
    # structural parameters (7)
    'psi', 'mu_w', 'sigma', 'mu_s', 'gamma', 'alpha', 'beta',
    # implied multiplier (1)
    'eta',
    # bootstrap standard errors (6) -- no se_psi, by design
    'se_mu_w', 'se_sigma', 'se_mu_s', 'se_gamma', 'se_alpha', 'se_beta',
    # cost of moral hazard (3)
    'delta_1', 'delta_2', 'delta_3',
    # bootstrap standard errors on the costs (3)
    'se_delta_1', 'se_delta_2', 'se_delta_3',
    # counterfactual costs (3) -- point estimates only, no SEs
    'cf_gamma_delta_1', 'cf_gamma_delta_2', 'cf_gamma_delta_3',
]
assert len(SUBMISSION_IDS) == 23


def make_submission(res: Dict, grid: Dict, path: str = 'submission.csv') -> str:
    """Write submission.csv: header 'ID,ESTIMATE', one row per
    SUBMISSION_IDS entry, in that fixed order.

    res  -- flat dict of the baseline structural-parameter estimates,
            eta, their bootstrap SEs, and the three cost-decomposition
            deltas with their bootstrap SEs (20 of the 23 IDs).
    grid -- dict keyed by counterfactual scenario name ('gamma'),
            with a 'delta_1'/'delta_2'/'delta_3' point estimate (the
            remaining 3 IDs, flattened as cf_<scenario>_delta_<k>).
    """
    values = {}
    for k in ['psi', 'mu_w', 'sigma', 'mu_s', 'gamma', 'alpha', 'beta', 'eta',
              'se_mu_w', 'se_sigma', 'se_mu_s', 'se_gamma', 'se_alpha', 'se_beta',
              'delta_1', 'delta_2', 'delta_3',
              'se_delta_1', 'se_delta_2', 'se_delta_3']:
        values[k] = res[k]
    for scenario in ['gamma']:
        for k in ['delta_1', 'delta_2', 'delta_3']:
            values[f'cf_{scenario}_{k}'] = grid[scenario][k]

    missing = [i for i in SUBMISSION_IDS if i not in values]
    if missing:
        raise ValueError(f"make_submission: missing required IDs: {missing}")
    bad = [i for i in SUBMISSION_IDS if not np.isfinite(values[i])]
    if bad:
        raise ValueError(f"make_submission: non-finite estimate(s) for: {bad}")

    out_df = pd.DataFrame({'ID': SUBMISSION_IDS,
                            'ESTIMATE': [values[i] for i in SUBMISSION_IDS]})
    out_df.to_csv(path, index=False)
    return path


def validate_submission(path: str = 'submission.csv') -> None:
    """Re-read the written submission and check header, ID set, row order,
    and that every ESTIMATE parses as a finite float."""
    df = pd.read_csv(path)
    assert list(df.columns) == ['ID', 'ESTIMATE'], f"bad header: {list(df.columns)}"
    assert list(df['ID']) == SUBMISSION_IDS, "row order/ID set does not match SUBMISSION_IDS"
    vals = df['ESTIMATE'].astype(float)
    assert np.all(np.isfinite(vals)), "non-finite ESTIMATE value(s) in submission"
    print(f"validate_submission: OK -- {len(df)} rows, header and ID order correct, all finite.")

In [ ]:
res = {
    'psi': model.psi, 'mu_w': model.mu_w_hat, 'sigma': model.sigma_hat,
    'mu_s': model.mu_s_hat, 'gamma': model.gamma_hat, 'alpha': model.alpha_hat,
    'beta': model.beta_hat, 'eta': model.eta_hat,
    'se_mu_w': boot['se_mu_w'], 'se_sigma': boot['se_sigma'], 'se_mu_s': boot['se_mu_s'],
    'se_gamma': boot['se_gamma'], 'se_alpha': boot['se_alpha'], 'se_beta': boot['se_beta'],
    'delta_1': costs['Delta1'], 'delta_2': costs['Delta2'], 'delta_3': costs['Delta3'],
    'se_delta_1': boot['se_delta_1'], 'se_delta_2': boot['se_delta_2'], 'se_delta_3': boot['se_delta_3'],
}

path = make_submission(res, grid)
validate_submission(path)